In [0]:
%sql
-- Databricks notebook source
DROP TABLE IF EXISTS retail_etl.silver_customers;

CREATE TABLE retail_etl.silver_customers AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY CustomerID
               ORDER BY LastUpdated DESC
           ) rn
    FROM retail_etl.bronze_customers
)
SELECT
    CustomerID,
    INITCAP(TRIM(CustomerName)) AS CustomerName,
    LOWER(TRIM(Email)) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    LastUpdated
FROM ranked
WHERE rn = 1;

-- COMMAND ----------

DROP TABLE IF EXISTS retail_etl.rejected_products;

CREATE TABLE retail_etl.rejected_products AS
SELECT *
FROM retail_etl.bronze_products
WHERE UnitPrice <= 0;

-- COMMAND ----------

DROP TABLE IF EXISTS retail_etl.silver_products;

CREATE TABLE retail_etl.silver_products AS
SELECT
    ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    UnitPrice
FROM retail_etl.bronze_products
WHERE UnitPrice > 0;

-- COMMAND ----------

DROP TABLE IF EXISTS retail_etl.silver_stores;

CREATE TABLE retail_etl.silver_stores AS
SELECT
    StoreID,
    INITCAP(TRIM(StoreName)) AS StoreName,
    COALESCE(NULLIF(TRIM(Region), ''), 'Unknown') AS Region
FROM retail_etl.bronze_stores;

-- COMMAND ----------

DROP TABLE IF EXISTS retail_etl.sales_stage;

CREATE TABLE retail_etl.sales_stage AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY TransactionID
               ORDER BY TxnDate DESC
           ) rn
    FROM retail_etl.bronze_sales
)
SELECT
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity,
    TxnDate
FROM ranked
WHERE rn = 1;

-- COMMAND ----------

DROP TABLE IF EXISTS retail_etl.rejected_sales_qty;

CREATE TABLE retail_etl.rejected_sales_qty AS
SELECT *
FROM retail_etl.sales_stage
WHERE Quantity <= 0;

-- COMMAND ----------

DROP TABLE IF EXISTS retail_etl.sales_valid_qty;

CREATE TABLE retail_etl.sales_valid_qty AS
SELECT *
FROM retail_etl.sales_stage
WHERE Quantity > 0;

-- COMMAND ----------

DROP TABLE IF EXISTS retail_etl.silver_sales;

CREATE TABLE retail_etl.silver_sales AS
SELECT
    TransactionID,
    CustomerID,
    CASE
        WHEN ProductID = 101 THEN 1001
        WHEN ProductID = 102 THEN 1002
        WHEN ProductID = 103 THEN 1003
        ELSE ProductID
    END AS ProductID,
    StoreID,
    Quantity,
    TxnDate
FROM retail_etl.sales_valid_qty;